# Chapter 9.2 - Converting Raw Text into Sequence Data

Models consume numbers, while language arrives as characters and words. This notebook builds the complete text pipeline: normalize text, tokenize it, construct a vocabulary, map tokens to integer IDs, and inspect language statistics.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. Everything is generated from small tensors or inline text, so there are no downloads. Before important cells, predict the time, batch, feature, vocabulary, and hidden-state shapes. Treat every assertion as an executable contract rather than decoration.

## You are done when you can

- distinguish a text line, token, token ID, vocabulary, and corpus
- tokenize the same text at word and character granularity
- read and extend a small vocabulary class
- round-trip known tokens through IDs
- handle unknown tokens deliberately instead of silently failing


In [ ]:
import math
import random
from collections import Counter

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
random.seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)


## 9.2.0 The Problem This Notebook Solves

Raw text is not yet training data. A sequence model needs a deterministic mapping:

```text
raw text -> cleaned lines -> tokens -> vocabulary -> integer token IDs
```

A **token** is the unit the model sees, such as a word or character. A **vocabulary** is the finite lookup table between tokens and integer IDs. A **corpus** is the complete token-ID sequence used as data.

Integer IDs are names, not measurements. Token ID 8 is not twice token ID 4. Later code will convert IDs to one-hot vectors or learned embeddings before treating them as numeric features.


## 9.2.1 Reading and Normalizing an Inline Dataset

Real D2L code can download a book and read it line by line. This notebook uses an original inline corpus so it is reproducible offline. Normalization lowercases letters, replaces punctuation with spaces, collapses repeated whitespace, and discards blank lines.


In [ ]:
import re

RAW_TEXT = (
    "Signals arrive in order; models remember useful context.\n"
    "Context changes a prediction, and prediction changes the next context!\n"
    "Small sequences make hidden mechanics visible."
)

def read_inline_text(text):
    cleaned = []
    for raw_line in text.strip().splitlines():
        line = re.sub("[^A-Za-z]+", " ", raw_line).strip().lower()
        if line:
            cleaned.append(line)
    return cleaned

lines = read_inline_text(RAW_TEXT)
print(lines)
assert len(lines) == 3
assert all(line == line.lower() for line in lines)
assert all(";" not in line and "!" not in line for line in lines)


## 9.2.2 Tokenization: Choose the Unit of Sequence

**Word tokenization** splits a line into word units. **Character tokenization** splits it into individual characters, including spaces if we keep them. The choice changes sequence length and vocabulary size.

Word tokens make sequences shorter but need an unknown-token policy for unseen words. Character tokens use a small vocabulary but require many more time steps to express the same sentence.


In [ ]:
def tokenize(lines, mode="word"):
    if mode == "word":
        return [line.split() for line in lines]
    if mode == "char":
        return [list(line) for line in lines]
    raise ValueError("mode must be 'word' or 'char'")

word_tokens = tokenize(lines, "word")
char_tokens = tokenize(lines, "char")
print("words:", word_tokens[0])
print("characters:", char_tokens[0][:12])
assert len(char_tokens[0]) > len(word_tokens[0])
assert "signals" in word_tokens[0]


## 9.2.3 Vocabulary: A Small Inspectable Python Class

The class below stores two inverse mappings:

- `idx_to_token[id]` returns a token;
- `token_to_idx[token]` returns an ID.

`self` is the instance being operated on. `__init__` runs when `Vocab(...)` constructs that instance. `__len__` lets Python call `len(vocab)`. `__getitem__` lets square brackets call `vocab[tokens]`. These are ordinary methods connected to familiar Python syntax.

`<unk>` is the unknown token. Reserved tokens are inserted before corpus tokens so their IDs are stable.


In [ ]:
class Vocab:
    def __init__(self, token_lines, min_freq=1, reserved_tokens=None):
        reserved_tokens = reserved_tokens or []
        counts = Counter(token for line in token_lines for token in line)
        self.token_freqs = sorted(counts.items(), key=lambda pair: (-pair[1], pair[0]))
        self.idx_to_token = ["<unk>"]
        for token in reserved_tokens:
            if token not in self.idx_to_token:
                self.idx_to_token.append(token)
        for token, frequency in self.token_freqs:
            if frequency >= min_freq and token not in self.idx_to_token:
                self.idx_to_token.append(token)
        self.token_to_idx = {token: index for index, token in enumerate(self.idx_to_token)}

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        if isinstance(tokens, str):
            return self.token_to_idx.get(tokens, self.unk)
        return [self[token] for token in tokens]

    @property
    def unk(self):
        return 0

    def to_tokens(self, indices):
        if isinstance(indices, int):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]


The `@property` decorator makes the zero-argument method `unk` readable as `vocab.unk` instead of `vocab.unk()`. It does not precompute a new value; Python calls the method when the attribute is accessed.

Before the next cell, predict the ID of `<unk>` and what ID the unseen word `memory` receives.


In [ ]:
vocab = Vocab(word_tokens, min_freq=1, reserved_tokens=["<pad>"])
known = ["context", "prediction"]
known_ids = vocab[known]
round_trip = vocab.to_tokens(known_ids)

print(list(enumerate(vocab.idx_to_token)))
print("known round trip:", known, known_ids, round_trip)
print("unseen token ID:", vocab["memory"])
assert vocab["<unk>"] == vocab.unk == 0
assert vocab["<pad>"] == 1
assert round_trip == known
assert vocab["memory"] == vocab.unk


## 9.2.4 Putting It All Together: Build a Corpus

Flattening removes line boundaries and creates one long stream. Keeping a parallel token list makes the alignment inspectable. Every corpus element is an integer in `[0, len(vocab))`.


In [ ]:
flat_tokens = [token for line in word_tokens for token in line]
corpus = torch.tensor(vocab[flat_tokens], dtype=torch.long)
decoded = vocab.to_tokens(corpus.tolist())

print("corpus IDs:", corpus)
print("decoded prefix:", decoded[:8])
assert corpus.ndim == 1
assert len(corpus) == len(flat_tokens)
assert decoded == flat_tokens
assert 0 <= corpus.min() and corpus.max() < len(vocab)


## 9.2.5 Exploratory Language Statistics

A **unigram** is one token. A **bigram** is an ordered pair of adjacent tokens. A **trigram** contains three. Their frequency tables reveal both common units and local order.

Natural-language frequencies are often highly uneven: a few tokens occur frequently and many are rare. Small corpora are too noisy for broad conclusions, but the counting mechanism is the same at scale.


In [ ]:
unigrams = Counter(flat_tokens)
bigrams = Counter(zip(flat_tokens[:-1], flat_tokens[1:]))
trigrams = Counter(zip(flat_tokens[:-2], flat_tokens[1:-1], flat_tokens[2:]))

print("top unigrams:", unigrams.most_common(5))
print("top bigrams:", bigrams.most_common(5))
print("top trigrams:", trigrams.most_common(3))
assert sum(unigrams.values()) == len(flat_tokens)
assert sum(bigrams.values()) == len(flat_tokens) - 1
assert sum(trigrams.values()) == len(flat_tokens) - 2


## 9.2.6 Break It Deliberately: No Unknown-Token Policy

A raw dictionary lookup raises `KeyError` for an unseen token. That failure is safer than silently assigning an arbitrary ID, but production text pipelines normally reserve an explicit unknown ID so inference can continue predictably.


In [ ]:
unsafe_mapping = {"known": 0}
try:
    unsafe_mapping["never_seen"]
except KeyError as error:
    print(type(error).__name__, error)
else:
    raise AssertionError("Expected an unseen token to raise KeyError")

assert vocab["never_seen"] == vocab.unk


## 9.2 Checkpoint

Answer these without rerunning the notebook. Short markdown answers are enough.

1. Why are integer token IDs names rather than numeric measurements?
2. How does character tokenization trade sequence length for vocabulary size?
3. What state is stored on each Vocab instance through self?
4. Why reserve an unknown token?
5. How many bigrams exist in a flattened sequence of N tokens?
